# Residual-Stream Norm Audit: is "1x" the same push everywhere?

**No ESMFold in this notebook. No folding, no generation-quality scoring.** It only runs forward
passes and measures the size of each model's internal hidden state. That makes it the cheapest
run in the project (~20-35 minutes) and it should be run before any further steering work.

## The problem this exists to settle

Every fair-test notebook (`24`, `26`, `27`, `28`, `32`) rescales its steering vector to the same
**absolute** reference norm, `REFERENCE_NORM = 583.998`, and calls that "1x". That was introduced
as Fix 2 to correct a real confound: `09` injected each vector at *its own* raw norm, so layer 30
got a 1256.69-sized push while layer 12 got a 584-sized one, and the resulting "deeper layers are
more fragile" finding turned out to be an artifact of the unequal push (retracted in `25`).

Fix 2 equalized the **absolute** push. It did not equalize the **relative** push. Two things
follow, and neither has ever been checked:

**1. Across models, 584 is not one push.** The raw utility-matched `v_L` norms actually recorded
in the five fair-test notebooks span more than four orders of magnitude:

| Model | early raw ‖v_L‖ | late raw ‖v_L‖ | early scale-up to reach 584 | late scale-up |
|---|---|---|---|---|
| ProtGPT2 (L12/L30) | 439.6188 | 781.1768 | 1.33x | 0.75x |
| ZymCTRL (L12/L30) | 3.3572 | 5.9438 | 174x | 98x |
| p-IgGen (L1/L3) | 0.5670 | 1.8319 | **1030x** | 319x |
| Mistral-Prot (L2/L7) | 3664.0 | 5280.0 | 0.159x | 0.111x |
| RITA (L3/L11) | 2.9786 | 5.3513 | 196x | 109x |

Note ProtGPT2 (439.6) and ZymCTRL (3.36) are the same architecture family with identical
1280-dim / 36-layer shape, yet sit 130x apart -- so this is not a hidden-dimension effect, it is
a genuine per-model difference in internal scale.

**2. Within every model, the early layer is pushed harder in relative terms.** The late layer's
raw norm is larger than the early layer's in **all five models** (by 1.44x to 3.23x). That is the
expected behaviour of a residual stream, which accumulates as depth increases. Setting both to the
same absolute 584 therefore over-pushes the shallower layer by roughly that factor -- *in exactly
the direction of the reported "early layers are more sensitive, 4-for-4" trend.* And the ordering
is uncomfortable: p-IgGen has both the largest asymmetry (3.23x) and by far the largest observed
gap (100% vs 32%), and it is the only model where that gap is statistically significant.

## What this notebook measures

The principled denominator is not the contrast vector's size, it is **the size of the thing the
vector is added to**: the residual stream itself. This notebook records, for every layer of every
model, the mean per-position hidden-state norm, then reports

    alpha_rel(layer) = ‖v_inject‖ / ‖h(layer)‖

which is the actual fraction by which the hidden state is perturbed. Under `alpha_rel`, "1x" finally
means the same thing across layers and across models.

Two methodological choices worth stating, both deliberate:

- **Measurement is taken with forward hooks on each block, not from `output_hidden_states`.** Hooks
  capture exactly the tensor the steering hook modifies, which is the quantity we care about. This
  also sidesteps RITA's non-standard `output_hidden_states` behaviour, so one code path covers all
  five models.
- **Per-position norm, not pooled norm.** The steering hook broadcasts `v` onto *every* token
  position, so the relevant comparison is ‖v‖ against a typical single position's ‖h_i‖, i.e.
  `mean_i ‖h_i‖`. The pooled quantity `‖mean_i h_i‖` is also recorded for reference, since that is
  what `v_L` itself is built from, but it is not the right denominator here.

**Off-by-one note, found while writing this and worth confirming independently.** The fair-test
notebooks build `v_L` from `out.hidden_states[L]` but inject via a hook on block `L`. In HuggingFace
causal LMs `hidden_states[L]` is the *input* to block `L` (equivalently the output of block `L-1`),
while a forward hook on block `L` fires on its *output*. So the vector is read one block earlier than
it is injected. It is a one-block offset, probably minor, but this notebook measures norms at both
positions so the size of the discrepancy is visible rather than assumed.

## How to read the result

- If `alpha_rel(early) / alpha_rel(late)` is close to 1.0 in every model, the layer comparison was
  fair after all and the early>late trend stands, now properly defended.
- If that ratio is meaningfully above 1.0 -- which the raw `v_L` norms above predict -- then early
  layers were simply pushed harder, and the trend must be re-examined or retracted, exactly as `09`
  was by `25`.

Either outcome is publishable and neither threatens the steering arm's central claim (more push ->
more collapse). What is at stake is only the *layer-sensitivity* sub-claim.

Kaggle setup: Accelerator = **GPU T4 x1**, Internet = **ON**. Expect ~20-35 minutes.


In [3]:
import warnings
warnings.filterwarnings("ignore")

import gc
import json
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(2024)
np.random.seed(2024)

device = "cuda" if torch.cuda.is_available() else "cpu"

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

# The single shared reference norm used by every fair-test notebook (24/26/27/28/32).
REFERENCE_NORM = 583.998

print("Setup complete. CUDA available:", torch.cuda.is_available())


Setup complete. CUDA available: True


In [4]:
# --- A single common probe set, used identically for all five models. ---
#
# Using the SAME real protein fragments for every model is what makes the cross-model
# comparison clean: if each model were measured on its own generated output instead, a model
# that generates degenerate sequences would be measured in a different part of its own input
# space than a model that generates well, and the norms would not be comparable. Every model
# here can encode a plain amino-acid string in a forward pass, even the ones that would not
# naturally *generate* that way (ZymCTRL is EC-conditioned, p-IgGen is antibody-only) -- and a
# forward pass is all this measurement needs.
#
# Fragment length 50 matches this project's standard generation budget (max_len=50).

UNIPROT_ACCESSIONS = [
    "P0CG48", "P00720", "P02144", "P42212", "P01308", "P61823",
    "P00648", "P99999", "P69905", "P68871", "P00698", "P00441",
]

def fetch_uniprot_sequence(accession, timeout=10):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            text = resp.read().decode("utf-8")
        lines = [l for l in text.strip().split("\n") if l]
        seq = "".join(lines[1:])
        return seq if len(seq) >= 20 else None
    except Exception as e:
        print(f"  skip {accession}: {e}")
        return None

print("Fetching real reference protein sequences from UniProt...")
reference_seqs = []
for acc in UNIPROT_ACCESSIONS:
    seq = fetch_uniprot_sequence(acc)
    if seq:
        reference_seqs.append((acc, seq))
        print(f"  fetched {acc}: {len(seq)} residues")

def build_probe_set(reference_seqs, n_probes=40, frag_len=50, seed=7):
    rng = np.random.RandomState(seed)
    probes = []
    for i in range(n_probes):
        acc, seq = reference_seqs[i % len(reference_seqs)]
        start = rng.randint(0, max(1, len(seq) - frag_len))
        probes.append(seq[start:start + frag_len])
    return probes

N_PROBES = 40
probe_seqs = build_probe_set(reference_seqs, n_probes=N_PROBES)
print(f"\nBuilt {len(probe_seqs)} common probe fragments of ~50 residues.")
print(f"Example: {probe_seqs[0]}")


Fetching real reference protein sequences from UniProt...
  fetched P0CG48: 685 residues
  fetched P00720: 164 residues
  fetched P02144: 154 residues
  fetched P42212: 238 residues
  fetched P01308: 110 residues
  fetched P61823: 150 residues
  fetched P00648: 157 residues
  fetched P99999: 105 residues
  fetched P69905: 142 residues
  fetched P68871: 147 residues
  fetched P00698: 147 residues
  fetched P00441: 154 residues

Built 40 common probe fragments of ~50 residues.
Example: ENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRL


In [5]:
# --- Per-model configuration. Hook paths and tested layers are copied verbatim from each
#     model's own fair-test notebook so this audit measures exactly the injection sites those
#     experiments used. Raw v_L norms are transcribed from those notebooks' saved output cells
#     (notes/locked-results.md SS1g-SS1k) -- reproduced here only to compute the
#     "scale-up applied" column, not re-derived. ---

MODEL_CONFIGS = [
    {"name": "ProtGPT2",          "hf_id": "nferruz/ProtGPT2",
     "path": "transformer.h",     "layers": [12, 30], "source_nb": "24",
     "raw_vL": {12: 439.6188, 30: 781.1768}, "loader": "standard"},

    {"name": "ZymCTRL",           "hf_id": "AI4PD/ZymCTRL",
     "path": "transformer.h",     "layers": [12, 30], "source_nb": "26",
     "raw_vL": {12: 3.3572, 30: 5.9438}, "loader": "standard"},

    {"name": "p-IgGen",           "hf_id": "opig/p-IgGen",
     "path": "gpt_neox.layers",   "layers": [1, 3],   "source_nb": "27",
     "raw_vL": {1: 0.5670, 3: 1.8319}, "loader": "standard"},

    {"name": "Mistral-Prot-134M", "hf_id": "RaphaelMourad/Mistral-Prot-v1-134M",
     "path": "model.layers",      "layers": [2, 7],   "source_nb": "28",
     "raw_vL": {2: 3664.0, 7: 5280.0}, "loader": "standard"},

    {"name": "RITA-small",        "hf_id": "lightonai/RITA_s",
     "path": "transformer.layers", "layers": [3, 11], "source_nb": "32",
     "raw_vL": {3: 2.9786, 11: 5.3513}, "loader": "rita"},
]

def resolve_layer_list(model, dotted_path):
    obj = model
    for part in dotted_path.split("."):
        obj = getattr(obj, part)
    return obj

# --- Scoped RITA loader, unchanged from 32-ai4dd-rita-fair-test.ipynb. Scoping matters even
#     here: an unscoped patch would leave PreTrainedModel permanently modified and could break
#     any later model load in the same session (context-and-decisions SS9). ---

def load_rita(device):
    import transformers.modeling_utils as _mu
    _had_mark_tied = "mark_tied_weights_as_initialized" in _mu.PreTrainedModel.__dict__
    _orig_mark_tied = _mu.PreTrainedModel.__dict__.get("mark_tied_weights_as_initialized")
    _had_all_tied_keys = "all_tied_weights_keys" in _mu.PreTrainedModel.__dict__
    _orig_all_tied_keys = _mu.PreTrainedModel.__dict__.get("all_tied_weights_keys")
    _mu.PreTrainedModel.mark_tied_weights_as_initialized = lambda self: None
    _mu.PreTrainedModel.all_tied_weights_keys = property(lambda self: {})
    try:
        tokenizer = AutoTokenizer.from_pretrained("lightonai/RITA_s")
        model = AutoModelForCausalLM.from_pretrained(
            "lightonai/RITA_s", trust_remote_code=True, torch_dtype=torch.float32
        ).to(device)
        model.eval()
    finally:
        if _had_mark_tied:
            _mu.PreTrainedModel.mark_tied_weights_as_initialized = _orig_mark_tied
        else:
            del _mu.PreTrainedModel.mark_tied_weights_as_initialized
        if _had_all_tied_keys:
            _mu.PreTrainedModel.all_tied_weights_keys = _orig_all_tied_keys
        else:
            del _mu.PreTrainedModel.all_tied_weights_keys
    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token_id is not None:
            tokenizer.pad_token = tokenizer.eos_token
        else:
            tokenizer.add_special_tokens({"pad_token": "[PAD]"})
            model.resize_token_embeddings(len(tokenizer))
    return tokenizer, model

def load_model(cfg, device):
    if cfg["loader"] == "rita":
        return load_rita(device)
    tokenizer = AutoTokenizer.from_pretrained(cfg["hf_id"])
    model = AutoModelForCausalLM.from_pretrained(cfg["hf_id"]).to(device)
    model.eval()
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer, model

print(f"{len(MODEL_CONFIGS)} models configured for audit.")


5 models configured for audit.


In [6]:
# --- The measurement itself. One forward hook per block captures that block's output --
#     precisely the tensor a steering hook would add v to -- and we record two norms:
#
#       per_pos : mean over token positions of ‖h_i‖   <- the correct denominator, since the
#                                                         steering hook broadcasts v onto every
#                                                         position independently
#       pooled  : ‖mean over positions of h_i‖         <- recorded only because this is the
#                                                         quantity v_L itself is built from
#
#     Also captured: the norm read from output_hidden_states[L], to quantify the one-block
#     read/inject offset noted in the header. Skipped automatically for RITA, whose
#     output_hidden_states does not return the standard tuple. ---

def measure_layer_norms(model, tokenizer, seqs, layer_list, device, want_hidden_states=True):
    n_layers = len(layer_list)
    per_pos_sums = np.zeros(n_layers)
    pooled_sums = np.zeros(n_layers)
    counted = np.zeros(n_layers)
    captured = {}

    def make_hook(idx):
        def hook(module, inp, out):
            h = out[0] if isinstance(out, (tuple, list)) else out
            captured[idx] = h.detach()
        return hook

    handles = [layer_list[i].register_forward_hook(make_hook(i)) for i in range(n_layers)]

    hs_per_pos_sums = np.zeros(n_layers + 1)
    hs_counted = np.zeros(n_layers + 1)

    try:
        for seq in seqs:
            enc = tokenizer(seq, return_tensors="pt", truncation=True, max_length=256)
            input_ids = enc["input_ids"].to(device)
            if input_ids.shape[1] < 2:
                continue
            captured.clear()
            with torch.no_grad():
                if want_hidden_states:
                    try:
                        out = model(input_ids=input_ids, output_hidden_states=True)
                        hs = getattr(out, "hidden_states", None)
                    except Exception:
                        hs = None
                else:
                    model(input_ids=input_ids)
                    hs = None

                if want_hidden_states and hs is None:
                    model(input_ids=input_ids)

            for i in range(n_layers):
                if i not in captured:
                    continue
                h = captured[i].float()
                per_pos_sums[i] += h.norm(dim=-1).mean().item()
                pooled_sums[i] += h.mean(dim=1).norm().item()
                counted[i] += 1

            if hs is not None and isinstance(hs, (tuple, list)):
                for i, layer_h in enumerate(hs):
                    if i > n_layers:
                        break
                    lh = layer_h.float()
                    hs_per_pos_sums[i] += lh.norm(dim=-1).mean().item()
                    hs_counted[i] += 1
    finally:
        for hd in handles:
            hd.remove()

    per_pos = {i: (per_pos_sums[i] / counted[i]) for i in range(n_layers) if counted[i] > 0}
    pooled = {i: (pooled_sums[i] / counted[i]) for i in range(n_layers) if counted[i] > 0}
    hs_norms = {i: (hs_per_pos_sums[i] / hs_counted[i]) for i in range(n_layers + 1) if hs_counted[i] > 0}
    return per_pos, pooled, hs_norms

print("Measurement function ready.")


Measurement function ready.


In [ ]:
# --- Main loop: load each model, measure every layer on the common probe set, free it, move on.
#     Models are loaded one at a time and explicitly deleted so a single T4 is enough. ---

all_results = {}

for cfg in MODEL_CONFIGS:
    name = cfg["name"]
    print(f"\n{'=' * 70}")
    print(f"=== {name}  ({cfg['hf_id']})")
    print(f"{'=' * 70}")
    try:
        tokenizer, model = load_model(cfg, device)
        layer_list = resolve_layer_list(model, cfg["path"])
        n_layers = len(layer_list)
        hidden_size = getattr(model.config, "hidden_size", getattr(model.config, "n_embd", None))
        print(f"  {n_layers} blocks at model.{cfg['path']}, hidden_size={hidden_size}")

        per_pos, pooled, hs_norms = measure_layer_norms(
            model, tokenizer, probe_seqs, layer_list, device,
            want_hidden_states=(cfg["loader"] != "rita"),
        )

        all_results[name] = {
            "config": {k: v for k, v in cfg.items() if k != "raw_vL"},
            "raw_vL": cfg["raw_vL"],
            "n_layers": n_layers,
            "hidden_size": hidden_size,
            "per_pos": per_pos,
            "pooled": pooled,
            "hidden_states": hs_norms,
        }

        print(f"\n  Residual-stream norm by depth (mean per-position ‖h‖ over {len(probe_seqs)} probes):")
        for i in sorted(per_pos):
            marker = "  <== TESTED" if i in cfg["layers"] else ""
            if n_layers <= 12 or i in cfg["layers"] or i % max(1, n_layers // 8) == 0:
                print(f"    block {i:2d}: {per_pos[i]:12.2f}{marker}")

        del model, tokenizer
        clear_gpu()
        print(f"  {name} freed from GPU.")
    except Exception as e:
        print(f"  !! FAILED on {name}: {type(e).__name__}: {e}")
        print(f"  Continuing to next model -- partial results are still usable.")
        try:
            del model, tokenizer
        except Exception:
            pass
        clear_gpu()

print(f"\n\nMeasured {len(all_results)}/{len(MODEL_CONFIGS)} models successfully.")



=== ProtGPT2  (nferruz/ProtGPT2)


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

In [12]:
# --- Result 1: is "1x" the same relative push across models? ---
#
# alpha_rel = ‖v_inject‖ / ‖h‖ is the fraction by which each token's hidden state is displaced.
# Under the current REFERENCE_NORM convention ‖v_inject‖ is always 583.998 at "1x", so any
# variation in alpha_rel across models or layers is variation in how hard they were actually pushed.

rows = []
for name, res in all_results.items():
    for layer in res["config"]["layers"]:
        h = res["per_pos"].get(layer)
        if h is None:
            continue
        raw_vL = res["raw_vL"][layer]
        rows.append({
            "model": name,
            "layer": layer,
            "n_layers": res["n_layers"],
            "rel_depth": layer / max(1, res["n_layers"] - 1),
            "resid_norm_h": h,
            "raw_vL_norm": raw_vL,
            "scaleup_to_584": REFERENCE_NORM / raw_vL,
            "alpha_rel_1x": REFERENCE_NORM / h,
            "alpha_rel_2x": 2 * REFERENCE_NORM / h,
        })

df = pd.DataFrame(rows).sort_values(["model", "layer"])

print("=" * 104)
print("ALPHA_REL: how large the injected vector actually is, relative to the hidden state it is added to")
print("=" * 104)
print(f"{'Model':20s} {'Layer':>6s} {'RelDepth':>9s} {'‖h‖':>12s} {'raw‖v_L‖':>11s} "
      f"{'scale-up':>10s} {'a_rel 1x':>10s} {'a_rel 2x':>10s}")
print("-" * 104)
for _, r in df.iterrows():
    print(f"{r['model']:20s} {int(r['layer']):6d} {r['rel_depth']:9.2f} {r['resid_norm_h']:12.2f} "
          f"{r['raw_vL_norm']:11.3f} {r['scaleup_to_584']:9.2f}x {r['alpha_rel_1x']:10.3f} "
          f"{r['alpha_rel_2x']:10.3f}")

print()
if len(df):
    lo, hi = df["alpha_rel_1x"].min(), df["alpha_rel_1x"].max()
    print(f"alpha_rel at nominal '1x' ranges {lo:.3f} -> {hi:.3f} across all tested model/layer pairs.")
    print(f"Spread factor: {hi / lo:.1f}x")
    if hi / lo > 2.0:
        print()
        print("  ==> CONFIRMED: '1x' is NOT one push. Conditions labelled identically across the")
        print("      cross-model table correspond to substantially different real perturbations.")
        print("      The cross-model steering comparison (locked-results SS1g-SS1k) needs to be")
        print("      re-expressed in alpha_rel units before it can be read as a like-for-like")
        print("      comparison.")
    else:
        print()
        print("  ==> The absolute-norm convention turns out to be a reasonable proxy for matched")
        print("      relative push here. The cross-model comparison stands as originally reported.")


ALPHA_REL: how large the injected vector actually is, relative to the hidden state it is added to
Model                 Layer  RelDepth          ‖h‖    raw‖v_L‖   scale-up   a_rel 1x   a_rel 2x
--------------------------------------------------------------------------------------------------------
Mistral-Prot-134M         2      0.29     56439.23    3664.000      0.16x      0.010      0.021
Mistral-Prot-134M         7      1.00    252032.61    5280.000      0.11x      0.002      0.005
ProtGPT2                 12      0.34      2921.53     439.619      1.33x      0.200      0.400
ProtGPT2                 30      0.86      3751.99     781.177      0.75x      0.156      0.311
RITA-small                3      0.27        23.69       2.979    196.06x     24.652     49.304
RITA-small               11      1.00        43.03       5.351    109.13x     13.572     27.143
ZymCTRL                  12      0.34        51.14       3.357    173.95x     11.419     22.839
ZymCTRL                  30  

In [13]:
# --- Result 2: the one that decides the "early layers are more sensitive" claim. ---
#
# Within each model, was the early layer pushed harder in relative terms than the late layer?
# asymmetry = alpha_rel(early) / alpha_rel(late) = ‖h(late)‖ / ‖h(early)‖
#
#   ~1.0  -> the two layers really did receive matched pushes; the trend is about the layers.
#   >1.0  -> the early layer was pushed harder, and "early layers are more sensitive" may be
#            partly or wholly a restatement of "early layers were pushed harder".

print("=" * 100)
print("WITHIN-MODEL LAYER FAIRNESS: was the early layer over-pushed relative to the late layer?")
print("=" * 100)
print(f"{'Model':20s} {'early':>6s} {'late':>6s} {'‖h‖ early':>12s} {'‖h‖ late':>12s} "
      f"{'a_rel early':>12s} {'a_rel late':>12s} {'asymmetry':>11s}")
print("-" * 100)

asymmetries = {}
for name, res in all_results.items():
    layers = res["config"]["layers"]
    if len(layers) < 2:
        continue
    early, late = layers[0], layers[1]
    h_e, h_l = res["per_pos"].get(early), res["per_pos"].get(late)
    if h_e is None or h_l is None:
        continue
    a_e, a_l = REFERENCE_NORM / h_e, REFERENCE_NORM / h_l
    asym = a_e / a_l
    asymmetries[name] = asym
    print(f"{name:20s} {early:6d} {late:6d} {h_e:12.2f} {h_l:12.2f} {a_e:12.3f} {a_l:12.3f} "
          f"{asym:10.2f}x")

print()
if asymmetries:
    vals = np.array(list(asymmetries.values()))
    n_over = int((vals > 1.15).sum())
    print(f"Models where the early layer received a >15% larger relative push: {n_over}/{len(vals)}")
    print(f"Median asymmetry: {np.median(vals):.2f}x")
    print()
    if n_over >= len(vals) - 1 and np.median(vals) > 1.15:
        print("  ==> CONFOUND CONFIRMED. The early layer was systematically pushed harder in")
        print("      relative terms in essentially every model. The '4-for-4 early > late'")
        print("      trend in locked-results SS1i/SS1k cannot be attributed to layer depth as")
        print("      long as it is reported at matched ABSOLUTE norm.")
        print()
        print("      Required action: either (a) re-run the early/late comparison at matched")
        print("      alpha_rel using the per-model scale factors printed below, or (b) retract")
        print("      the layer-sensitivity sub-claim in the same way SS1f was retracted by 25.")
        print("      The steering arm's main claim (more push -> more collapse) is unaffected")
        print("      either way -- only the layer sub-claim depends on this.")
    elif np.median(vals) > 1.15:
        print("  ==> PARTIAL CONFOUND. Some models were over-pushed at the early layer and some")
        print("      were not. Report per-model rather than as a single pooled trend.")
    else:
        print("  ==> NO CONFOUND DETECTED. The layers received comparable relative pushes, so the")
        print("      early > late trend is about depth, not about push size. The sub-claim is now")
        print("      defended against its strongest available objection.")

print()
print("-" * 100)
print("REPLACEMENT SCALE FACTORS -- to re-run any condition at genuinely matched relative push")
print("-" * 100)
print("Pick a single target alpha_rel (the ProtGPT2 layer-12 value is the natural anchor, since")
print("every headline number in this project is calibrated to it), then scale each vector to")
print("‖v‖ = target_alpha_rel * ‖h(layer)‖ instead of to a fixed 583.998.")
print()
anchor = None
if "ProtGPT2" in all_results and 12 in all_results["ProtGPT2"]["per_pos"]:
    anchor = REFERENCE_NORM / all_results["ProtGPT2"]["per_pos"][12]
    print(f"Suggested anchor: alpha_rel = {anchor:.4f}  (ProtGPT2 layer 12 at the current '1x')")
    print()
    print(f"{'Model':20s} {'Layer':>6s} {'‖h‖':>12s} {'matched ‖v‖ for 1x':>20s} {'vs current 584':>16s}")
    print("-" * 100)
    for name, res in all_results.items():
        for layer in res["config"]["layers"]:
            h = res["per_pos"].get(layer)
            if h is None:
                continue
            matched = anchor * h
            print(f"{name:20s} {layer:6d} {h:12.2f} {matched:20.2f} {matched / REFERENCE_NORM:15.2f}x")


WITHIN-MODEL LAYER FAIRNESS: was the early layer over-pushed relative to the late layer?
Model                 early   late    ‖h‖ early     ‖h‖ late  a_rel early   a_rel late   asymmetry
----------------------------------------------------------------------------------------------------
ProtGPT2                 12     30      2921.53      3751.99        0.200        0.156       1.28x
ZymCTRL                  12     30        51.14        82.66       11.419        7.065       1.62x
p-IgGen                   1      3         6.14        21.81       95.079       26.774       3.55x
Mistral-Prot-134M         2      7     56439.23    252032.61        0.010        0.002       4.47x
RITA-small                3     11        23.69        43.03       24.652       13.572       1.82x

Models where the early layer received a >15% larger relative push: 5/5
Median asymmetry: 1.82x

  ==> CONFOUND CONFIRMED. The early layer was systematically pushed harder in
      relative terms in essentially every

In [14]:
# --- Result 3: the read/inject one-block offset, quantified. ---
#
# The fair-test notebooks read v_L from out.hidden_states[L] but inject at a forward hook on
# block L. hidden_states[L] is the INPUT to block L; the hook fires on its OUTPUT. This cell
# reports how much the residual stream actually changes across that one block, so the size of
# the discrepancy is a measured number rather than an assumption.

print("=" * 92)
print("READ vs INJECT SITE: hidden_states[L] (read) is the input to block L; the hook (inject)")
print("fires on its output. How different are they in magnitude?")
print("=" * 92)
print(f"{'Model':20s} {'Layer':>6s} {'read ‖h‖':>14s} {'inject ‖h‖':>14s} {'ratio':>10s}")
print("-" * 92)

any_hs = False
for name, res in all_results.items():
    hs = res.get("hidden_states", {})
    if not hs:
        continue
    any_hs = True
    for layer in res["config"]["layers"]:
        read_norm = hs.get(layer)
        inject_norm = res["per_pos"].get(layer)
        if read_norm is None or inject_norm is None:
            continue
        print(f"{name:20s} {layer:6d} {read_norm:14.2f} {inject_norm:14.2f} "
              f"{inject_norm / read_norm:9.3f}x")

if not any_hs:
    print("(no hidden_states captured -- expected for RITA only; check other models loaded)")

print()
print("A ratio near 1.0 means the offset is cosmetic. A ratio far from 1.0 means the vector is")
print("being built in a measurably different place from where it is applied, which is worth one")
print("sentence in Methods either way.")


READ vs INJECT SITE: hidden_states[L] (read) is the input to block L; the hook (inject)
fires on its output. How different are they in magnitude?
Model                 Layer       read ‖h‖     inject ‖h‖      ratio
--------------------------------------------------------------------------------------------
ProtGPT2                 12        2761.20        2921.53     1.058x
ProtGPT2                 30        3630.12        3751.99     1.034x
ZymCTRL                  12          49.84          51.14     1.026x
ZymCTRL                  30          78.37          82.66     1.055x
p-IgGen                   1           3.60           6.14     1.709x
p-IgGen                   3          11.17          21.81     1.953x
Mistral-Prot-134M         2       61527.11       56439.23     0.917x
Mistral-Prot-134M         7       47994.50      252032.61     5.251x

A ratio near 1.0 means the offset is cosmetic. A ratio far from 1.0 means the vector is
being built in a measurably different place from wh

In [15]:
# --- Persist everything. ---
#
# Standing problem in this project: no notebook except 00a saves structured data, so every
# analysis needs a fresh GPU run. This writes the full per-layer profile for all five models to
# CSV/JSON so any follow-up (re-plotting, alpha_rel re-derivation, the depth-profile figure)
# can be done locally with no GPU at all.

profile_rows = []
for name, res in all_results.items():
    for i, h in sorted(res["per_pos"].items()):
        profile_rows.append({
            "model": name,
            "block": i,
            "n_layers": res["n_layers"],
            "rel_depth": i / max(1, res["n_layers"] - 1),
            "hidden_size": res["hidden_size"],
            "resid_norm_per_pos": h,
            "resid_norm_pooled": res["pooled"].get(i, float("nan")),
            "hidden_states_norm": res.get("hidden_states", {}).get(i, float("nan")),
            "was_tested": i in res["config"]["layers"],
            "alpha_rel_1x": REFERENCE_NORM / h,
        })

profile_df = pd.DataFrame(profile_rows)
profile_df.to_csv("residual_norm_profile.csv", index=False)
df.to_csv("alpha_rel_tested_layers.csv", index=False)

with open("residual_norm_audit_raw.json", "w") as f:
    json.dump({k: {kk: (vv if not isinstance(vv, dict) else {str(a): b for a, b in vv.items()})
                   for kk, vv in v.items()} for k, v in all_results.items()}, f, indent=2, default=str)

print("Saved:")
print(f"  residual_norm_profile.csv     ({len(profile_df)} rows -- every block of every model)")
print(f"  alpha_rel_tested_layers.csv   ({len(df)} rows -- just the tested layers)")
print(f"  residual_norm_audit_raw.json  (full raw record)")
print()
print("Download all three from Kaggle's output pane and commit them alongside this notebook.")
print()
print("=" * 78)
print("NEXT STEP")
print("=" * 78)
print("Paste the two tables above into notes/locked-results.md under the SS1k-FLAG section.")
print("Whichever way the verdict lands, it resolves an open, already-documented correctness")
print("question -- and if the confound is confirmed, the replacement scale factors printed in")
print("Result 2 are exactly what a matched-alpha_rel re-run needs.")


Saved:
  residual_norm_profile.csv     (96 rows -- every block of every model)
  alpha_rel_tested_layers.csv   (10 rows -- just the tested layers)
  residual_norm_audit_raw.json  (full raw record)

Download all three from Kaggle's output pane and commit them alongside this notebook.

NEXT STEP
Paste the two tables above into notes/locked-results.md under the SS1k-FLAG section.
Whichever way the verdict lands, it resolves an open, already-documented correctness
question -- and if the confound is confirmed, the replacement scale factors printed in
Result 2 are exactly what a matched-alpha_rel re-run needs.
